# Google Ads — revisão de 27/09/2026

Reprodução dos agregados revisados. As fontes vivas foram lidas antes da captura; este notebook não reconecta plataformas nem contém dados pessoais. Comparação observacional, sem causalidade nem maturação equivalente.

In [1]:
from pathlib import Path
import json, sqlite3
folder = Path.cwd()
if not (folder / 'ANALISE.json').exists(): folder = folder / 'auditorias/google-ads-revisao-2026-09-27'
data = json.loads((folder/'ANALISE.json').read_text(encoding='utf-8'))
artifact = json.loads((folder/'artifact.json').read_text(encoding='utf-8'))
print('Captura:', data['capturedAt'], 'Campanhas:', len(data['campaigns']))

Captura: 2026-09-27T12:39:49.686Z Campanhas: 16


In [2]:
for total in data['totals']:
    rows = [r for r in data['campaigns'] if r['period'] == total['period']]
    for key in ['impressions','clicks','cost','contacts','qualified','scheduled','completed','closed']:
        assert abs(sum(r[key] for r in rows)-total[key]) < 0.011, (key,total['period'])
    print(total['periodLabel'], 'R$',total['cost'], 'contatos',total['contacts'], 'qualificados',total['qualified'], 'consultas',total['scheduled'])
old = [r for r in data['campaigns'] if r['campaign'] not in ['G26MAMA','G26CORP']]
before=sum(r['cost'] for r in old if r['period']=='antes'); after=sum(r['cost'] for r in old if r['period']=='depois')
print('Campanhas antigas: aumento',round((after/before-1)*100,2),'%')

31/08–12/09 R$ 1127.76 contatos 6 qualificados 1 consultas 0
14–26/09 R$ 1362.38 contatos 7 qualificados 2 consultas 0
Campanhas antigas: aumento 2.1 %


## Consultas locais do relatório

SQLite recebe as capturas JSON revisadas. A query documentada em cada fonte deve produzir exatamente as linhas exibidas, sem fingir ser uma consulta ao Google Ads ou à LEADS.

In [3]:
con=sqlite3.connect(':memory:')
con.row_factory=sqlite3.Row
con.execute('CREATE TABLE audit_snapshot(payload TEXT)')
con.execute('INSERT INTO audit_snapshot VALUES (?)',(json.dumps(data),))
con.execute('CREATE TABLE report_snapshot(payload TEXT)')
con.execute('INSERT INTO report_snapshot VALUES (?)',(json.dumps(artifact['snapshot']['datasets']),))
sources={s['id']:s for s in artifact['manifest']['sources']}
for item in artifact['manifest']['charts']+artifact['manifest']['tables']:
    actual=[dict(r) for r in con.execute(sources[item['sourceId']]['query']['sql'])]
    expected=artifact['snapshot']['datasets'][item['dataset']]
    assert actual==expected, item['id']
    print(item['id'],len(actual),'linhas verificadas')
con.close()

campaign_cost 16 linhas verificadas
comparison_table 2 linhas verificadas
campaign_table 8 linhas verificadas
actions_table 4 linhas verificadas


## Limitações

Qualificação é fase atual da oportunidade, não renda presumida. Ausência de marco de consulta não comprova ausência de atendimento fora do registro. Exportações históricas misturam definições de conversão; o comparativo principal usa a interface recente. Termos divulgados cobrem somente parte do gasto. Sem inferência causal por amostra pequena e mudanças concorrentes no atendimento.